# RealBasicVSR Anime V3 — Kaggle Resume Training

Notebook này tiếp tục trực tiếp run Colab hiện tại.

**Add Input trước khi chạy:**
- `dataset_final_3.tar.gz`
- `last.pth` (ưu tiên resume)
- `best.pth` và `history.csv` (nên có)

Cấu hình được giữ như run hiện tại: 7 frame/window, crop 256, 2520 train windows/epoch, Adam, LR main `2e-5`, SPyNet `2e-6`, 20 epoch.

## 1. Kiểm tra GPU và Kaggle Input
Liệt kê GPU cùng các file đã Add Input.

In [ ]:
from pathlib import Path
import os, sys, math, random, time, csv, json, shutil, tarfile
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg19, VGG19_Weights
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("="*90)
print("REALBASICVSR ANIME V3 - KAGGLE RESUME")
print("="*90)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")

print("\n--- /kaggle/input ---")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_file():
        print(p)


In [ ]:
print("\n--- /kaggle/input ---")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_file():
        print(p)

## 2. Giải nén dataset
Tự tìm `dataset_final_3.tar.gz`, giải nén vào SSD `/kaggle/working` và xác định `train_GT/val_GT/test_GT`.

## 3. Audit dataset
Phải có 180/40/56 clip và 2520/560/784 temporal windows.

# nếu đã giải nén xong thì

In [ ]:
from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/datasets/duonghuyle/realbasicvsr-anime-data-v1/dataset_final_3"
)

TRAIN_DIR = DATA_ROOT / "train_GT"
VAL_DIR   = DATA_ROOT / "val_GT"
TEST_DIR  = DATA_ROOT / "test_GT"

print("DATA_ROOT:", DATA_ROOT)
print("TRAIN    :", TRAIN_DIR)
print("VAL      :", VAL_DIR)
print("TEST     :", TEST_DIR)

assert TRAIN_DIR.exists()
assert VAL_DIR.exists()
assert TEST_DIR.exists()

print("✓ Dataset path OK")

## 4. Kiến trúc RealBasicVSR
Định nghĩa đúng mạng của run hiện tại: cleaning → SPyNet → backward/forward propagation → fusion → PixelShuffle ×4.

In [ ]:
class ResidualBlockNoBN(nn.Module):
    def __init__(self, mid_channels=64, res_scale=1.0):
        super().__init__()
        self.res_scale=res_scale
        self.conv1=nn.Conv2d(mid_channels,mid_channels,3,1,1,bias=True)
        self.conv2=nn.Conv2d(mid_channels,mid_channels,3,1,1,bias=True)
        self.relu=nn.ReLU(inplace=True)
    def forward(self,x):
        return x+self.conv2(self.relu(self.conv1(x)))*self.res_scale

class ResidualBlocksWithInputConv(nn.Module):
    def __init__(self,in_channels,out_channels=64,num_blocks=20):
        super().__init__()
        self.main=nn.Sequential(
            nn.Conv2d(in_channels,out_channels,3,1,1,bias=True),
            nn.LeakyReLU(0.1,inplace=True),
            nn.Sequential(*[ResidualBlockNoBN(out_channels) for _ in range(num_blocks)])
        )
    def forward(self,x): return self.main(x)

class PixelShufflePack(nn.Module):
    def __init__(self,in_channels,out_channels,scale_factor=2,upsample_kernel=3):
        super().__init__()
        self.scale_factor=scale_factor
        self.upsample_conv=nn.Conv2d(
            in_channels,out_channels*scale_factor*scale_factor,
            upsample_kernel,padding=(upsample_kernel-1)//2
        )
    def forward(self,x):
        return F.pixel_shuffle(self.upsample_conv(x),self.scale_factor)

class ConvModule(nn.Module):
    def __init__(self,ic,oc,k=7,s=1,p=3,act=True):
        super().__init__()
        self.conv=nn.Conv2d(ic,oc,k,s,p,bias=True)
        self.act=nn.ReLU(inplace=False) if act else None
    def forward(self,x):
        x=self.conv(x)
        return self.act(x) if self.act is not None else x

class SPyNetBasicModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.basic_module=nn.Sequential(
            ConvModule(8,32),ConvModule(32,64),ConvModule(64,32),
            ConvModule(32,16),ConvModule(16,2,act=False)
        )
    def forward(self,x): return self.basic_module(x)

class SPyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.basic_module=nn.ModuleList([SPyNetBasicModule() for _ in range(6)])
        self.register_buffer("mean",torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std",torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
    def preprocess(self,x): return (x-self.mean)/self.std
    def flow_warp(self,x,flow,interpolation="bilinear",padding_mode="zeros"):
        n,c,h,w=x.size()
        gy,gx=torch.meshgrid(
            torch.arange(h,device=x.device,dtype=x.dtype),
            torch.arange(w,device=x.device,dtype=x.dtype),indexing="ij"
        )
        grid=torch.stack((gx,gy),2).unsqueeze(0).repeat(n,1,1,1)
        gf=grid+flow.permute(0,2,3,1)
        gxs=2.0*gf[...,0]/max(w-1,1)-1.0
        gys=2.0*gf[...,1]/max(h-1,1)-1.0
        return F.grid_sample(
            x,torch.stack((gxs,gys),dim=3),mode=interpolation,
            padding_mode=padding_mode,align_corners=True
        )
    def forward(self,ref,supp):
        n,c,h,w=ref.size()
        ref=[self.preprocess(ref)]; supp=[self.preprocess(supp)]
        for _ in range(5):
            ref.append(F.avg_pool2d(ref[-1],2,2,count_include_pad=False))
            supp.append(F.avg_pool2d(supp[-1],2,2,count_include_pad=False))
        ref,supp=ref[::-1],supp[::-1]
        flow=ref[0].new_zeros(n,2,ref[0].size(2),ref[0].size(3))
        for level in range(len(ref)):
            flow_up=flow if level==0 else F.interpolate(flow,scale_factor=2,mode="bilinear",align_corners=True)*2.0
            flow_res=self.basic_module[level](torch.cat([ref[level],self.flow_warp(supp[level],flow_up),flow_up],dim=1))
            flow=flow_up+flow_res
        return flow

class RealBasicVSRNet(nn.Module):
    def __init__(self,mid_channels=64,num_blocks=20):
        super().__init__()
        self.mid_channels=mid_channels
        self.image_cleaning=nn.Sequential(
            ResidualBlocksWithInputConv(3,64,20),
            nn.Conv2d(64,3,3,1,1,bias=True)
        )
        self.spynet=SPyNet()
        self.backward_resblocks=ResidualBlocksWithInputConv(mid_channels+3,mid_channels,num_blocks)
        self.forward_resblocks=ResidualBlocksWithInputConv(mid_channels+3,mid_channels,num_blocks)
        self.fusion=nn.Conv2d(mid_channels*2,mid_channels,1,1)
        self.upsample1=PixelShufflePack(mid_channels,mid_channels,2,3)
        self.upsample2=PixelShufflePack(mid_channels,64,2,3)
        self.conv_hr=nn.Conv2d(64,64,3,1,1)
        self.conv_last=nn.Conv2d(64,3,3,1,1)
        self.img_upsample=nn.Upsample(scale_factor=4,mode="bilinear",align_corners=False)
        self.lrelu=nn.LeakyReLU(0.1,inplace=True)
    def forward(self,lqs):
        n,t,c,h,w=lqs.size()
        flat=lqs.reshape(-1,c,h,w)
        clean=(self.image_cleaning(flat)+flat).clamp(0,1).reshape(n,t,c,h,w)

        ff=[]; fb=[]
        for i in range(t-1):
            ff.append(self.spynet(clean[:,i+1],clean[:,i]))
            fb.append(self.spynet(clean[:,i],clean[:,i+1]))

        backward=[]
        feat=lqs.new_zeros(n,self.mid_channels,h,w)
        for i in range(t-1,-1,-1):
            if i<t-1: feat=self.spynet.flow_warp(feat,fb[i])
            feat=feat+self.backward_resblocks(torch.cat([clean[:,i],feat],dim=1))
            backward.append(feat)
        backward=backward[::-1]

        outs=[]
        feat=lqs.new_zeros(n,self.mid_channels,h,w)
        for i in range(t):
            if i>0: feat=self.spynet.flow_warp(feat,ff[i-1])
            feat=feat+self.forward_resblocks(torch.cat([clean[:,i],feat],dim=1))
            out=self.fusion(torch.cat([backward[i],feat],dim=1))
            out=self.lrelu(self.upsample1(out))
            out=self.lrelu(self.upsample2(out))
            out=self.lrelu(self.conv_hr(out))
            out=self.conv_last(out)+self.img_upsample(lqs[:,i])
            outs.append(out)
        return torch.stack(outs,dim=1)

print("✓ Architecture ready")


## 5. Degradation + Full-window Dataset
Tạo LQ từ GT và dùng đủ 14 window/clip. Validation cố định để so sánh qua epoch.

In [ ]:
# ============================================================
# GLOBAL CONFIG
# Phải nằm trước Dataset / DataLoader / Model / Training
# ============================================================

NUM_FRAME = 7
TEMPORAL_STRIDE = 1
GT_CROP = 256

print("NUM_FRAME       :", NUM_FRAME)
print("TEMPORAL_STRIDE :", TEMPORAL_STRIDE)
print("GT_CROP         :", GT_CROP)

In [ ]:
GT_CROP=256
BATCH_SIZE=1

def get_gaussian_kernel2d(k=5,sx=1.0,sy=1.0,theta=0.0,device="cuda"):
    d=(k-1)/2.0
    coords=torch.arange(k,device=device,dtype=torch.float32)-d
    y,x=torch.meshgrid(coords,coords,indexing="ij")
    rx=math.cos(theta)*x+math.sin(theta)*y
    ry=-math.sin(theta)*x+math.cos(theta)*y
    k2d=torch.exp(-0.5*((rx**2)/(sx**2)+(ry**2)/(sy**2)))
    return k2d/k2d.sum()

def apply_chroma_subsample(x):
    r,g,b=x[:,0:1],x[:,1:2],x[:,2:3]
    y=0.299*r+0.587*g+0.114*b
    cb=-0.1687*r-0.3313*g+0.5*b+0.5
    cr=0.5*r-0.4187*g-0.0813*b+0.5
    h,w=cb.shape[2:]
    cb=F.interpolate(F.interpolate(cb,(h//2,w//2),mode="bilinear",align_corners=False),(h,w),mode="bilinear",align_corners=False)
    cr=F.interpolate(F.interpolate(cr,(h//2,w//2),mode="bilinear",align_corners=False),(h,w),mode="bilinear",align_corners=False)
    return torch.clamp(torch.cat([
        y+1.402*(cr-0.5),
        y-0.3441*(cb-0.5)-0.7141*(cr-0.5),
        y+1.772*(cb-0.5)
    ],dim=1),0,1)

@torch.no_grad()
def degrade_anime_gpu(gt,is_train=True,seed=None):
    B,T,C,H,W=gt.shape
    x=gt.reshape(B*T,C,H,W)
    device=x.device

    if (not is_train) or random.random()<0.85:
        sx=0.8 if not is_train else random.uniform(0.4,1.6)
        sy=0.8 if not is_train else random.uniform(0.4,1.6)
        th=0.0 if not is_train else random.uniform(0,math.pi)
        k2d=get_gaussian_kernel2d(5,sx,sy,th,device)
        ker=k2d.reshape(1,1,5,5).repeat(C,1,1,1)
        x=F.conv2d(x,ker,padding=2,groups=C)

    mode="bicubic" if not is_train else random.choice(["bicubic","bilinear","area"])
    x=F.interpolate(x,scale_factor=0.25,mode=mode) if mode=="area" else F.interpolate(x,scale_factor=0.25,mode=mode,align_corners=False)

    if (not is_train) or random.random()<0.5:
        x=apply_chroma_subsample(x)

    if (not is_train) or random.random()<0.7:
        sn=0.015 if not is_train else random.uniform(0.005,0.025)
        if is_train or seed is None:
            noise=torch.randn_like(x)
        else:
            gen=torch.Generator(device=device); gen.manual_seed(int(seed))
            noise=torch.randn(x.shape,generator=gen,device=device,dtype=x.dtype)
        x=x+noise*sn

    return x.clamp(0,1).reshape(B,T,C,H//4,W//4)

class AnimeWindowDataset(Dataset):
    def __init__(self,root_dir,train=True,num_frame=7,stride=1,crop_size=256):
        self.root=Path(root_dir); self.train=train
        self.num_frame=num_frame; self.stride=stride; self.crop_size=crop_size
        self.samples=[]
        clips=sorted([p for p in self.root.iterdir() if p.is_dir()])
        for folder in clips:
            names=sorted([p.name for p in folder.glob("*.png")])
            for start in range(0,len(names)-num_frame+1,stride):
                self.samples.append((folder,names,start))
        print(f"{self.root.name}: {len(clips)} clips -> {len(self.samples)} windows")
    def __len__(self): return len(self.samples)
    def __getitem__(self,idx):
        folder,names,start=self.samples[idx]
        imgs=[]
        for name in names[start:start+self.num_frame]:
            bgr=cv2.imread(str(folder/name),cv2.IMREAD_COLOR)
            if bgr is None: raise RuntimeError(folder/name)
            imgs.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
        gt=np.stack(imgs,0)
        h,w=gt.shape[1:3]
        assert h>=self.crop_size and w>=self.crop_size
        if self.train:
            top=random.randint(0,h-self.crop_size); left=random.randint(0,w-self.crop_size)
            gt=gt[:,top:top+self.crop_size,left:left+self.crop_size]
            if random.random()<0.5:
                gt=np.ascontiguousarray(gt[:,:,::-1])
        else:
            top=(h-self.crop_size)//2; left=(w-self.crop_size)//2
            gt=gt[:,top:top+self.crop_size,left:left+self.crop_size]
        return torch.from_numpy(np.ascontiguousarray(gt)).permute(0,3,1,2).float()/255.0

train_set=AnimeWindowDataset(TRAIN_DIR,True,NUM_FRAME,TEMPORAL_STRIDE,GT_CROP)
val_set=AnimeWindowDataset(VAL_DIR,False,NUM_FRAME,TEMPORAL_STRIDE,GT_CROP)
assert len(train_set)==2520 and len(val_set)==560

train_loader=DataLoader(train_set,batch_size=1,shuffle=True,num_workers=2,pin_memory=True,persistent_workers=True)
val_loader=DataLoader(val_set,batch_size=1,shuffle=False,num_workers=1,pin_memory=True,persistent_workers=True)


## 6. Loss + PSNR-Y
Loss = Pixel L1 + 0.1 VGG + 0.05 Edge + 0.1 Temporal.

In [ ]:
class PerceptualVGG(nn.Module):
    def __init__(self):
        super().__init__()
        vgg=vgg19(weights=VGG19_Weights.DEFAULT).features
        self.slice=nn.Sequential(*[vgg[i] for i in range(16)]).eval()
        for p in self.slice.parameters(): p.requires_grad=False
        self.register_buffer("mean",torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std",torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
    def forward(self,x): return self.slice((x-self.mean)/self.std)

class AnimeVSRLoss(nn.Module):
    def __init__(self, device):
        super().__init__()

        self.vgg = PerceptualVGG().to(device)
        self.l1 = nn.L1Loss()

        kx = torch.tensor(
            [
                [-1, 0, 1],
                [-2, 0, 2],
                [-1, 0, 1]
            ],
            dtype=torch.float32
        ).view(1, 1, 3, 3)

        ky = torch.tensor(
            [
                [-1, -2, -1],
                [0,  0,  0],
                [1,  2,  1]
            ],
            dtype=torch.float32
        ).view(1, 1, 3, 3)

        self.register_buffer(
            "sobel_x",
            kx
        )

        self.register_buffer(
            "sobel_y",
            ky
        )

        # Đưa TOÀN BỘ loss module lên đúng device
        self.to(device)

    def edges(self, x):

        g = (
            0.299 * x[:, 0:1]
            + 0.587 * x[:, 1:2]
            + 0.114 * x[:, 2:3]
        )

        gx = F.conv2d(
            g,
            self.sobel_x,
            padding=1
        )

        gy = F.conv2d(
            g,
            self.sobel_y,
            padding=1
        )

        return torch.sqrt(
            gx ** 2
            + gy ** 2
            + 1e-6
        )

    def forward(self, sr, gt):

        B, T, C, H, W = sr.shape

        # 1. Pixel loss
        pix = self.l1(
            sr,
            gt
        )

        # 2. VGG perceptual loss
        ids = [
            1,
            T // 2,
            T - 2
        ]

        sv = sr[:, ids].reshape(
            -1,
            C,
            H,
            W
        )

        gv = gt[:, ids].reshape(
            -1,
            C,
            H,
            W
        )

        vgg = self.l1(
            self.vgg(sv),
            self.vgg(gv)
        )

        # 3. Edge loss
        sr_edges = self.edges(
            sr.reshape(
                -1,
                C,
                H,
                W
            )
        )

        gt_edges = self.edges(
            gt.reshape(
                -1,
                C,
                H,
                W
            )
        )

        edge = self.l1(
            sr_edges,
            gt_edges
        )

        # 4. Temporal loss
        temp = self.l1(
            sr[:, 1:]
            - sr[:, :-1],

            gt[:, 1:]
            - gt[:, :-1]
        )

        total = (
            pix
            + 0.1 * vgg
            + 0.05 * edge
            + 0.1 * temp
        )

        return (
            total,
            pix,
            vgg,
            edge,
            temp
        )

def calc_psnr_y(sr,gt):
    ys=0.299*sr[:,0]+0.587*sr[:,1]+0.114*sr[:,2]
    yg=0.299*gt[:,0]+0.587*gt[:,1]+0.114*gt[:,2]
    mse=torch.mean((ys-yg)**2).item()
    return 99.0 if mse<1e-10 else 20*math.log10(1.0/math.sqrt(mse))


## 7. Model + optimizer + resume checkpoint
Giữ nguyên scheduler của run Colab. Ưu tiên `last.pth`, fallback sang `best.pth`.

In [ ]:
SEED=1234
NUM_BLOCK=20
LR_MAIN=2e-5
LR_SPYNET=2e-6
EPOCHS=20
VAL_EVERY=1
SAVE_EVERY=2
GRAD_CLIP=1.0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

RUN=Path("/kaggle/working/runs_realbasicvsr_anime_v3")
RUN.mkdir(parents=True,exist_ok=True)
CKPT_LAST=RUN/"last.pth"; CKPT_BEST=RUN/"best.pth"; HISTORY=RUN/"history.csv"

net=RealBasicVSRNet(64,NUM_BLOCK).to(DEVICE)
main_params=[p for n,p in net.named_parameters() if not n.startswith("spynet.")]
spy_params=[p for n,p in net.named_parameters() if n.startswith("spynet.")]
optimizer=torch.optim.Adam(
    [{"params":main_params,"lr":LR_MAIN},{"params":spy_params,"lr":LR_SPYNET}],
    betas=(0.9,0.99)
)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,T_max=EPOCHS,eta_min=2e-6
)

def find_inputs(name): return sorted(Path("/kaggle/input").rglob(name))

if CKPT_LAST.exists():
    resume=CKPT_LAST
else:
    last=find_inputs("last.pth")
    best=find_inputs("best.pth")
    resume=last[0] if last else (best[0] if best else None)
assert resume is not None, "Không tìm thấy last.pth/best.pth"

if not CKPT_BEST.exists():
    b=find_inputs("best.pth")
    if b: shutil.copy2(b[0],CKPT_BEST)
if not HISTORY.exists():
    h=find_inputs("history.csv")
    if h: shutil.copy2(h[0],HISTORY)

ckpt=torch.load(resume,map_location="cpu",weights_only=False)
state=ckpt["net"] if isinstance(ckpt,dict) and "net" in ckpt else ckpt
missing,unexpected=net.load_state_dict(state,strict=False)
assert not missing and not unexpected, f"Checkpoint mismatch: missing={len(missing)}, unexpected={len(unexpected)}"

if isinstance(ckpt,dict) and "optimizer" in ckpt:
    optimizer.load_state_dict(ckpt["optimizer"])
    for st in optimizer.state.values():
        for k,v in st.items():
            if torch.is_tensor(v): st[k]=v.to(DEVICE)
if isinstance(ckpt,dict) and "scheduler" in ckpt:
    scheduler.load_state_dict(ckpt["scheduler"])

start_epoch=int(ckpt.get("epoch",0)) if isinstance(ckpt,dict) else 0
best_val_psnr=float(ckpt.get("best_val_psnr",ckpt.get("val_psnr",-float("inf")))) if isinstance(ckpt,dict) else -float("inf")
net=net.to(DEVICE)

print("Resume:",resume)
print("Completed epoch:",start_epoch)
print("Next epoch:",start_epoch+1)
print("Best PSNR:",best_val_psnr)
print("LR main:",optimizer.param_groups[0]["lr"])
print("LR SPyNet:",optimizer.param_groups[1]["lr"])


## 8. Sanity check + VGG loss
Kiểm tra shape và mean sáng trước khi chạy training lâu.

In [ ]:
net.eval()
gt=next(iter(train_loader)).to(DEVICE,non_blocking=True)
with torch.no_grad():
    lq=degrade_anime_gpu(gt,is_train=False,seed=SEED)
    sr=net(lq).clamp(0,1)

print("GT:",tuple(gt.shape),"LQ:",tuple(lq.shape),"SR:",tuple(sr.shape))
print(f"GT mean={gt.mean().item():.4f} | LQ mean={lq.mean().item():.4f} | SR mean={sr.mean().item():.4f}")
print(f"LQ/GT={lq.mean().item()/max(gt.mean().item(),1e-8):.4f}")
assert tuple(gt.shape)==(1,7,3,256,256)
assert tuple(lq.shape)==(1,7,3,64,64)
assert tuple(sr.shape)==(1,7,3,256,256)
del gt,lq,sr
torch.cuda.empty_cache()

loss_fn=AnimeVSRLoss(DEVICE)
print("✓ Sanity check + loss ready")


## 9. Validation, checkpoint và biểu đồ
Validation chạy đủ 560 windows. `last.pth` lưu mỗi epoch, `best.pth` lưu khi PSNR-Y tăng.

In [ ]:
FIELDS=["epoch","train_loss","loss_pixel","loss_vgg","loss_edge","loss_temporal","val_psnr_y","lr_main","lr_spynet","epoch_minutes"]

if not HISTORY.exists():
    pd.DataFrame(columns=FIELDS).to_csv(HISTORY,index=False)

@torch.no_grad()
def validate_model():
    net.eval(); vals=[]
    for idx,gt in enumerate(tqdm(val_loader,desc="Validation",leave=False)):
        gt=gt.to(DEVICE,non_blocking=True)
        lq=degrade_anime_gpu(gt,is_train=False,seed=SEED+idx)
        sr=net(lq).clamp(0,1)
        B,T,C,H,W=sr.shape
        vals.append(calc_psnr_y(sr.reshape(B*T,C,H,W),gt.reshape(B*T,C,H,W)))
    return float(np.mean(vals))

def save_history(row):
    try: df=pd.read_csv(HISTORY)
    except: df=pd.DataFrame(columns=FIELDS)
    if "epoch" in df.columns: df=df[df["epoch"]!=row["epoch"]]
    df=pd.concat([df,pd.DataFrame([row])],ignore_index=True).sort_values("epoch")
    df.to_csv(HISTORY,index=False)

def plot_history():
    df=pd.read_csv(HISTORY)
    if df.empty: return
    plt.figure(figsize=(10,6))
    for c,l in [("train_loss","Total"),("loss_pixel","Pixel"),("loss_vgg","VGG"),("loss_edge","Edge"),("loss_temporal","Temporal")]:
        if c in df and pd.to_numeric(df[c],errors="coerce").notna().any():
            plt.plot(df["epoch"],pd.to_numeric(df[c],errors="coerce"),marker="o",label=l)
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("RealBasicVSR Anime V3 - Training Loss")
    plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
    plt.savefig(RUN/"training_loss_v3.png",dpi=180,bbox_inches="tight"); plt.show(); plt.close()

    ps=df[pd.to_numeric(df["val_psnr_y"],errors="coerce").notna()].copy()
    if len(ps):
        ps["val_psnr_y"]=pd.to_numeric(ps["val_psnr_y"],errors="coerce")
        plt.figure(figsize=(10,6))
        plt.plot(ps["epoch"],ps["val_psnr_y"],marker="o")
        bi=ps["val_psnr_y"].idxmax()
        plt.scatter([ps.loc[bi,"epoch"]],[ps.loc[bi,"val_psnr_y"]],s=80,label=f"Best: {ps.loc[bi,'val_psnr_y']:.3f} dB")
        plt.xlabel("Epoch"); plt.ylabel("PSNR-Y (dB)"); plt.title("Validation PSNR-Y")
        plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
        plt.savefig(RUN/"validation_psnr_v3.png",dpi=180,bbox_inches="tight"); plt.show(); plt.close()


## 10. Train tiếp
Nếu checkpoint là epoch 7, cell này bắt đầu epoch 8. Nếu Colab dừng giữa epoch 8 thì chạy lại epoch 8 từ đầu là bình thường.

In [ ]:
print("="*90)
print("START / RESUME TRAINING")
print("="*90)
print("Completed:",start_epoch,"| Target:",EPOCHS,"| Steps/epoch:",len(train_loader))

for epoch_idx in range(start_epoch,EPOCHS):
    epoch=epoch_idx+1
    t0=time.time()
    net.train()
    sums=dict(total=0.0,pixel=0.0,vgg=0.0,edge=0.0,temporal=0.0)

    pbar=tqdm(train_loader,desc=f"Train {epoch:02d}/{EPOCHS}",leave=True)
    for gt in pbar:
        gt=gt.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        lq=degrade_anime_gpu(gt,is_train=True)
        sr=net(lq)
        loss,pix,vgg,edge,temp=loss_fn(sr,gt)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(),GRAD_CLIP)
        optimizer.step()

        sums["total"]+=loss.item(); sums["pixel"]+=pix.item(); sums["vgg"]+=vgg.item()
        sums["edge"]+=edge.item(); sums["temporal"]+=temp.item()

        pbar.set_postfix(loss=f"{loss.item():.4f}",pix=f"{pix.item():.4f}",lr=f"{optimizer.param_groups[0]['lr']:.1e}")

    n=len(train_loader)
    av={k:v/n for k,v in sums.items()}

    val_psnr=validate_model() if epoch%VAL_EVERY==0 else None
    scheduler.step()

    lr_main=optimizer.param_groups[0]["lr"]
    lr_spy=optimizer.param_groups[1]["lr"]
    mins=(time.time()-t0)/60

    is_best=val_psnr is not None and val_psnr>best_val_psnr
    if is_best: best_val_psnr=float(val_psnr)

    payload={
        "epoch":epoch,
        "net":net.state_dict(),
        "optimizer":optimizer.state_dict(),
        "scheduler":scheduler.state_dict(),
        "val_psnr":val_psnr,
        "best_val_psnr":best_val_psnr,
        "config":{
            "dataset":"dataset_final_3","num_frame":NUM_FRAME,"gt_crop":GT_CROP,
            "num_block":NUM_BLOCK,"lr_main":LR_MAIN,"lr_spynet":LR_SPYNET,"epochs":EPOCHS
        }
    }

    torch.save(payload,CKPT_LAST)
    if is_best: torch.save(payload,CKPT_BEST)
    if epoch%SAVE_EVERY==0: torch.save(payload,RUN/f"epoch_{epoch:03d}.pth")

    save_history({
        "epoch":epoch,"train_loss":av["total"],"loss_pixel":av["pixel"],"loss_vgg":av["vgg"],
        "loss_edge":av["edge"],"loss_temporal":av["temporal"],"val_psnr_y":val_psnr,
        "lr_main":lr_main,"lr_spynet":lr_spy,"epoch_minutes":mins
    })

    print("\n"+"-"*90)
    print(f"Epoch {epoch:02d}/{EPOCHS}")
    print(f"Train loss : {av['total']:.6f}")
    print(f"  Pixel    : {av['pixel']:.6f}")
    print(f"  VGG      : {av['vgg']:.6f}")
    print(f"  Edge     : {av['edge']:.6f}")
    print(f"  Temporal : {av['temporal']:.6f}")
    print(f"Val PSNR-Y : {val_psnr:.3f} dB" if val_psnr is not None else "Val PSNR-Y : skipped")
    print(f"LR main    : {lr_main:.3e}")
    print(f"LR SPyNet  : {lr_spy:.3e}")
    print(f"Time       : {mins:.1f} min")
    if is_best: print(f"✓ NEW BEST: {best_val_psnr:.3f} dB")
    print("-"*90)

    plot_history()
    torch.cuda.empty_cache()

print("\n✓ Training finished / target reached")


## 11. Output cuối
Kiểm tra checkpoint, CSV và biểu đồ. **Save Version** trên Kaggle hoặc tải file về sau khi train.

In [ ]:
print("Output:",RUN)
for p in sorted(RUN.glob("*")):
    if p.is_file():
        print(f"{p.name:28s} {p.stat().st_size/1024**2:9.2f} MB")
if HISTORY.exists():
    display(pd.read_csv(HISTORY).tail(10))


In [ ]:
import shutil

src = "/kaggle/working/runs_realbasicvsr_anime_v3"
out = "/kaggle/working/realbasicvsr_anime_v3_checkpoints"

shutil.make_archive(out, "zip", src)

print("Done:", out + ".zip")

# 12.Test

## Cell 1 — Cấu hình test

In [ ]:
from pathlib import Path
import os
import sys
import time
import shutil
import subprocess

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from IPython.display import display, Video


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

INPUT_ROOT = Path(
    "/kaggle/input/datasets/duonghuyle/realbasicvsr-anime-data-v1/dataset_final_3"
)


OUT_ROOT = Path(
    "/kaggle/working/vsr_benchmark"
)

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Test configuration
# ------------------------------------------------------------

SCALE = 4

# RealBasicVSR xử lý từng đoạn 7 frame.
CHUNK = 7

# Overlap giữa các đoạn để giảm artifact ở biên temporal.
OVERLAP = 2

# Video test thử trước 180 frames ~ 6-8 giây.
# Sau khi ổn có thể đổi None để chạy toàn video.
MAX_VIDEO_FRAMES = 180


print("DEVICE:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

## Cell 2 — Tìm checkpoint và testLQ

In [ ]:
from pathlib import Path

# ==============================================================================
# PATHS
# ==============================================================================

DATA_ROOT = Path(
    "/kaggle/input/datasets/duonghuyle/realbasicvsr-anime-data-v1/dataset_final_3"
)

WORK_ROOT = Path(
    "/kaggle/working"
)

RUN_ROOT = Path(
    "/kaggle/working/runs_realbasicvsr_anime_v3"
)


# ==============================================================================
# DATASET
# ==============================================================================

TEST_LQ = DATA_ROOT / "testLQ"

# Nếu dataset của bạn chưa có testLQ thì để None,
# lát nữa có thể dùng video upload riêng.
if not TEST_LQ.exists():
    TEST_LQ = None


# ==============================================================================
# FINE-TUNED MODEL
# ==============================================================================

FT_CKPT = RUN_ROOT / "best.pth"

if not FT_CKPT.exists():
    FT_CKPT = None


# ==============================================================================
# ORIGINAL REALBASICVSR
# ==============================================================================

# Nếu bạn chưa upload pretrained original vào Kaggle,
# để None và cell sau sẽ download.
RB_ORIGINAL_CKPT = None


# ==============================================================================
# ANIMESR
# ==============================================================================

# Nếu chưa Add Input AnimeSR_v2.pth thì để None.
ANIMESR_CKPT = None


# ==============================================================================
# PRINT
# ==============================================================================

print("DATA_ROOT     :", DATA_ROOT)
print("Fine-tuned   :", FT_CKPT)
print("RB original  :", RB_ORIGINAL_CKPT)
print("AnimeSR v2   :", ANIMESR_CKPT)
print("testLQ       :", TEST_LQ)


assert DATA_ROOT.exists(), (
    f"Không tìm thấy DATA_ROOT: {DATA_ROOT}"
)

assert FT_CKPT is not None, (
    "Không tìm thấy best.pth trong /kaggle/working/runs_realbasicvsr_anime_v3"
)

print("\n✓ Fine-tuned checkpoint found.")

## Cell 3 — Kiến trúc RealBasicVSR

In [ ]:
class ResidualBlockNoBN(nn.Module):
    def __init__(self, mid_channels=64, res_scale=1.0):
        super().__init__()
        self.res_scale=res_scale
        self.conv1=nn.Conv2d(mid_channels,mid_channels,3,1,1,bias=True)
        self.conv2=nn.Conv2d(mid_channels,mid_channels,3,1,1,bias=True)
        self.relu=nn.ReLU(inplace=True)
    def forward(self,x):
        return x+self.conv2(self.relu(self.conv1(x)))*self.res_scale

class ResidualBlocksWithInputConv(nn.Module):
    def __init__(self,in_channels,out_channels=64,num_blocks=20):
        super().__init__()
        self.main=nn.Sequential(
            nn.Conv2d(in_channels,out_channels,3,1,1,bias=True),
            nn.LeakyReLU(0.1,inplace=True),
            nn.Sequential(*[ResidualBlockNoBN(out_channels) for _ in range(num_blocks)])
        )
    def forward(self,x): return self.main(x)

class PixelShufflePack(nn.Module):
    def __init__(self,in_channels,out_channels,scale_factor=2,upsample_kernel=3):
        super().__init__()
        self.scale_factor=scale_factor
        self.upsample_conv=nn.Conv2d(
            in_channels,out_channels*scale_factor*scale_factor,
            upsample_kernel,padding=(upsample_kernel-1)//2
        )
    def forward(self,x):
        return F.pixel_shuffle(self.upsample_conv(x),self.scale_factor)

class ConvModule(nn.Module):
    def __init__(self,ic,oc,k=7,s=1,p=3,act=True):
        super().__init__()
        self.conv=nn.Conv2d(ic,oc,k,s,p,bias=True)
        self.act=nn.ReLU(inplace=False) if act else None
    def forward(self,x):
        x=self.conv(x)
        return self.act(x) if self.act is not None else x

class SPyNetBasicModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.basic_module=nn.Sequential(
            ConvModule(8,32),ConvModule(32,64),ConvModule(64,32),
            ConvModule(32,16),ConvModule(16,2,act=False)
        )
    def forward(self,x): return self.basic_module(x)

class SPyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.basic_module=nn.ModuleList([SPyNetBasicModule() for _ in range(6)])
        self.register_buffer("mean",torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std",torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
    def preprocess(self,x): return (x-self.mean)/self.std
    def flow_warp(self,x,flow,interpolation="bilinear",padding_mode="zeros"):
        n,c,h,w=x.size()
        gy,gx=torch.meshgrid(
            torch.arange(h,device=x.device,dtype=x.dtype),
            torch.arange(w,device=x.device,dtype=x.dtype),indexing="ij"
        )
        grid=torch.stack((gx,gy),2).unsqueeze(0).repeat(n,1,1,1)
        gf=grid+flow.permute(0,2,3,1)
        gxs=2.0*gf[...,0]/max(w-1,1)-1.0
        gys=2.0*gf[...,1]/max(h-1,1)-1.0
        return F.grid_sample(
            x,torch.stack((gxs,gys),dim=3),mode=interpolation,
            padding_mode=padding_mode,align_corners=True
        )
    def forward(self,ref,supp):
        n,c,h,w=ref.size()
        ref=[self.preprocess(ref)]; supp=[self.preprocess(supp)]
        for _ in range(5):
            ref.append(F.avg_pool2d(ref[-1],2,2,count_include_pad=False))
            supp.append(F.avg_pool2d(supp[-1],2,2,count_include_pad=False))
        ref,supp=ref[::-1],supp[::-1]
        flow=ref[0].new_zeros(n,2,ref[0].size(2),ref[0].size(3))
        for level in range(len(ref)):
            flow_up=flow if level==0 else F.interpolate(flow,scale_factor=2,mode="bilinear",align_corners=True)*2.0
            flow_res=self.basic_module[level](torch.cat([ref[level],self.flow_warp(supp[level],flow_up),flow_up],dim=1))
            flow=flow_up+flow_res
        return flow

class RealBasicVSRNet(nn.Module):
    def __init__(self,mid_channels=64,num_blocks=20):
        super().__init__()
        self.mid_channels=mid_channels
        self.image_cleaning=nn.Sequential(
            ResidualBlocksWithInputConv(3,64,20),
            nn.Conv2d(64,3,3,1,1,bias=True)
        )
        self.spynet=SPyNet()
        self.backward_resblocks=ResidualBlocksWithInputConv(mid_channels+3,mid_channels,num_blocks)
        self.forward_resblocks=ResidualBlocksWithInputConv(mid_channels+3,mid_channels,num_blocks)
        self.fusion=nn.Conv2d(mid_channels*2,mid_channels,1,1)
        self.upsample1=PixelShufflePack(mid_channels,mid_channels,2,3)
        self.upsample2=PixelShufflePack(mid_channels,64,2,3)
        self.conv_hr=nn.Conv2d(64,64,3,1,1)
        self.conv_last=nn.Conv2d(64,3,3,1,1)
        self.img_upsample=nn.Upsample(scale_factor=4,mode="bilinear",align_corners=False)
        self.lrelu=nn.LeakyReLU(0.1,inplace=True)
    def forward(self,lqs):
        n,t,c,h,w=lqs.size()
        flat=lqs.reshape(-1,c,h,w)
        clean=(self.image_cleaning(flat)+flat).clamp(0,1).reshape(n,t,c,h,w)

        ff=[]; fb=[]
        for i in range(t-1):
            ff.append(self.spynet(clean[:,i+1],clean[:,i]))
            fb.append(self.spynet(clean[:,i],clean[:,i+1]))

        backward=[]
        feat=lqs.new_zeros(n,self.mid_channels,h,w)
        for i in range(t-1,-1,-1):
            if i<t-1: feat=self.spynet.flow_warp(feat,fb[i])
            feat=feat+self.backward_resblocks(torch.cat([clean[:,i],feat],dim=1))
            backward.append(feat)
        backward=backward[::-1]

        outs=[]
        feat=lqs.new_zeros(n,self.mid_channels,h,w)
        for i in range(t):
            if i>0: feat=self.spynet.flow_warp(feat,ff[i-1])
            feat=feat+self.forward_resblocks(torch.cat([clean[:,i],feat],dim=1))
            out=self.fusion(torch.cat([backward[i],feat],dim=1))
            out=self.lrelu(self.upsample1(out))
            out=self.lrelu(self.upsample2(out))
            out=self.lrelu(self.conv_hr(out))
            out=self.conv_last(out)+self.img_upsample(lqs[:,i])
            outs.append(out)
        return torch.stack(outs,dim=1)

print("✓ Architecture ready")


## Cell 4 — Load RealBasicVSR original

In [ ]:
RB_URL = (
    "https://huggingface.co/"
    "akhaliq/RealBasicVSR_x4/"
    "resolve/main/"
    "RealBasicVSR_x4.pth"
)


if RB_ORIGINAL_CKPT is None:

    RB_ORIGINAL_CKPT = (
        OUT_ROOT /
        "RealBasicVSR_x4.pth"
    )

    if not RB_ORIGINAL_CKPT.exists():

        print(
            "Downloading "
            "RealBasicVSR original..."
        )

        subprocess.run(
            [
                "wget",
                "-q",
                "-O",
                str(RB_ORIGINAL_CKPT),
                RB_URL
            ],
            check=True
        )


def load_rb_original(path):

    net = RealBasicVSRNet(
        mid_channels=64,
        num_blocks=20
    )

    raw = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    w = raw.get(
        "state_dict",
        raw.get(
            "params",
            raw.get(
                "params_ema",
                raw
            )
        )
    )

    target = net.state_dict()


    # --------------------------------------------------------
    # RealBasicVSR original thường chứa generator / generator_ema
    # --------------------------------------------------------

    if any(
        k.startswith("generator_ema.")
        for k in w
    ):

        prefix = "generator_ema."

    else:

        prefix = "generator."


    mapped = {}


    for k, v in w.items():

        if not k.startswith(prefix):

            continue


        clean = k[len(prefix):]


        if clean.startswith(
            "basicvsr."
        ):

            clean = clean[
                len("basicvsr.") :
            ]


        if (
            clean in target
            and
            target[clean].shape
            ==
            v.shape
        ):

            mapped[clean] = v


    missing, unexpected = (
        net.load_state_dict(
            mapped,
            strict=False
        )
    )


    print(
        "Loaded original:",
        f"{len(mapped)}/{len(target)}"
    )

    print(
        "Missing:",
        len(missing)
    )

    print(
        "Unexpected:",
        len(unexpected)
    )


    assert len(mapped) == len(target)


    return (
        net
        .to(DEVICE)
        .eval()
    )


rb_original = load_rb_original(
    RB_ORIGINAL_CKPT
)

## Cell 5 — Load RealBasicVSR fine-tuned

In [ ]:
def load_rb_finetuned(path):

    net = RealBasicVSRNet(
        mid_channels=64,
        num_blocks=20
    )


    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )


    if (
        isinstance(ckpt, dict)
        and
        "net" in ckpt
    ):

        state = ckpt["net"]

    else:

        state = ckpt


    # Hỗ trợ checkpoint DataParallel
    if any(
        k.startswith("module.")
        for k in state
    ):

        state = {
            k.replace(
                "module.",
                "",
                1
            ): v

            for k, v
            in state.items()
        }


    net.load_state_dict(
        state,
        strict=True
    )


    if isinstance(
        ckpt,
        dict
    ):

        print(
            "Best epoch:",
            ckpt.get(
                "epoch"
            )
        )

        print(
            "Best Val PSNR:",
            ckpt.get(
                "best_val_psnr"
            )
        )


    return (
        net
        .to(DEVICE)
        .eval()
    )


rb_finetuned = (
    load_rb_finetuned(
        FT_CKPT
    )
)

## Cell 6 — Helper đọc ảnh + inference RealBasicVSR

Điểm đáng chú ý ở đây là không resize LQ xuống 160 hoặc 256 như notebook cũ.
Model nhận nguyên resolution của testLQ.
Chỉ pad biên để SPyNet xử lý an toàn.

In [ ]:
IMG_EXT = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".webp"
}


def read_rgb(path):

    img = cv2.imread(
        str(path),
        cv2.IMREAD_COLOR
    )

    if img is None:

        raise RuntimeError(
            f"Cannot read {path}"
        )


    return cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )


def save_rgb(
    path,
    img
):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    cv2.imwrite(
        str(path),
        cv2.cvtColor(
            img,
            cv2.COLOR_RGB2BGR
        )
    )

Padding:

In [ ]:
def pad_lq(x):

    # x:
    # B,T,C,H,W

    B, T, C, H, W = x.shape


    pad_h = (
        32 - H % 32
    ) % 32

    pad_w = (
        32 - W % 32
    ) % 32


    if (
        pad_h == 0
        and
        pad_w == 0
    ):

        return x, (H, W)


    flat = x.reshape(
        B * T,
        C,
        H,
        W
    )


    flat = F.pad(
        flat,
        (
            0,
            pad_w,
            0,
            pad_h
        ),
        mode="reflect"
    )


    return (
        flat.reshape(
            B,
            T,
            C,
            H + pad_h,
            W + pad_w
        ),
        (H, W)
    )

Inference theo temporal chunks:

In [ ]:
@torch.no_grad()
def infer_rb_sequence(
    net,
    frames,
    chunk=7,
    overlap=2
):

    N = len(frames)

    assert N > 0


    H, W = (
        frames[0]
        .shape[:2]
    )


    accum = [

        np.zeros(
            (
                H * 4,
                W * 4,
                3
            ),
            dtype=np.float32
        )

        for _ in range(N)
    ]


    count = np.zeros(
        N,
        dtype=np.float32
    )


    stride = max(
        1,
        chunk - overlap
    )


    starts = range(
        0,
        N,
        stride
    )


    for start in tqdm(
        starts,
        desc="RealBasicVSR"
    ):

        end = min(
            N,
            start + chunk
        )


        current = (
            frames[
                start:end
            ]
        )


        arr = (
            np.stack(
                current
            )
            .astype(
                np.float32
            )
            / 255.0
        )


        x = (
            torch
            .from_numpy(arr)
            .permute(
                0,
                3,
                1,
                2
            )
            .unsqueeze(0)
            .to(DEVICE)
        )


        x_pad, (
            original_h,
            original_w
        ) = pad_lq(x)


        sr = (
            net(x_pad)
            .clamp(
                0,
                1
            )
        )


        # crop bỏ padding
        sr = sr[
            :,
            :,
            :,
            :original_h * 4,
            :original_w * 4
        ]


        sr = (
            sr[0]
            .permute(
                0,
                2,
                3,
                1
            )
            .cpu()
            .numpy()
        )


        sr = (
            sr * 255.0
        ).round().astype(
            np.uint8
        )


        for j in range(
            end - start
        ):

            idx = (
                start + j
            )

            accum[idx] += (
                sr[j]
                .astype(
                    np.float32
                )
            )

            count[idx] += 1


        if end == N:

            break


    output = []


    for i in range(N):

        img = (
            accum[i]
            /
            max(
                count[i],
                1
            )
        )


        output.append(
            np.clip(
                img,
                0,
                255
            )
            .round()
            .astype(
                np.uint8
            )
        )


    return output

## Cell 7 — Setup AnimeSR official

In [ ]:
ANIMESR_REPO = (
    OUT_ROOT /
    "AnimeSR"
)


if not ANIMESR_REPO.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/"
            "TencentARC/AnimeSR.git",
            str(ANIMESR_REPO)
        ],
        check=True
    )

Cài:

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "basicsr==1.4.2",
        "facexlib",
        "gfpgan",
        "lmdb"
    ],
    check=False
)


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(ANIMESR_REPO),
        "--no-deps"
    ],
    check=False
)

Copy weight:

Nếu chưa có AnimeSR_v2.path thì dùng

In [ ]:
# ==============================================================================
# CELL 7 - SETUP ANIMESR OFFICIAL + UPLOAD AnimeSR_v2.pth
# ==============================================================================

from pathlib import Path
import subprocess
import sys
import shutil
import ipywidgets as widgets
from IPython.display import display


# ==============================================================================
# PATHS
# ==============================================================================

ANIMESR_REPO = Path(
    "/kaggle/working/AnimeSR"
)

ANIMESR_WEIGHTS_DIR = (
    ANIMESR_REPO
    / "weights"
)

ANIMESR_TARGET = (
    ANIMESR_WEIGHTS_DIR
    / "AnimeSR_v2.pth"
)

UPLOAD_DIR = Path(
    "/kaggle/working/uploads"
)

UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==============================================================================
# 1. CLONE OFFICIAL REPOSITORY
# ==============================================================================

if not ANIMESR_REPO.exists():

    print(
        "Cloning official TencentARC/AnimeSR..."
    )

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/TencentARC/AnimeSR.git",
            str(ANIMESR_REPO)
        ],
        check=True
    )

else:

    print(
        "✓ AnimeSR repo already exists"
    )


# ==============================================================================
# 2. INSTALL DEPENDENCIES
# ==============================================================================

print(
    "\nInstalling AnimeSR dependencies..."
)


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "basicsr==1.4.2",
        "facexlib",
        "gfpgan",
        "lmdb",
        "yapf"
    ],
    check=False
)


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(ANIMESR_REPO),
        "--no-deps"
    ],
    check=False
)


print(
    "✓ Dependencies installed"
)


# ==============================================================================
# 3. CREATE WEIGHTS DIRECTORY
# ==============================================================================

ANIMESR_WEIGHTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==============================================================================
# 4. CHECK IF WEIGHT ALREADY EXISTS
# ==============================================================================

if (
    ANIMESR_TARGET.exists()
    and
    ANIMESR_TARGET.stat().st_size
    > 1024 * 1024
):

    ANIMESR_CKPT = (
        ANIMESR_TARGET
    )

    print(
        "\n✓ AnimeSR_v2.pth already exists"
    )

    print(
        "Weight:",
        ANIMESR_CKPT
    )

    print(
        "Size:",
        f"{ANIMESR_CKPT.stat().st_size / 1024**2:.2f} MB"
    )


else:

    print(
        "\nAnimeSR_v2.pth chưa có."
    )

    print(
        "Upload file AnimeSR_v2.pth bên dưới:"
    )


    # ==========================================================================
    # 5. UPLOAD WIDGET
    # ==========================================================================

    uploader_animesr = (
        widgets.FileUpload(
            accept=".pth",
            multiple=False,
            description="Upload AnimeSR_v2.pth"
        )
    )


    upload_output = (
        widgets.Output()
    )


    # ==========================================================================
    # 6. CALLBACK SAVE FILE
    # ==========================================================================

    def save_animesr_weight(
        change
    ):

        global ANIMESR_CKPT


        if (
            len(
                uploader_animesr.value
            )
            == 0
        ):

            return


        with upload_output:

            upload_output.clear_output()


            # --------------------------------------------------------------
            # ipywidgets 7
            # --------------------------------------------------------------

            if isinstance(
                uploader_animesr.value,
                dict
            ):

                item = next(
                    iter(
                        uploader_animesr
                        .value
                        .values()
                    )
                )

                filename = (
                    item[
                        "metadata"
                    ][
                        "name"
                    ]
                )

                content = (
                    item[
                        "content"
                    ]
                )


            # --------------------------------------------------------------
            # ipywidgets 8
            # --------------------------------------------------------------

            else:

                item = (
                    uploader_animesr
                    .value[0]
                )

                filename = (
                    item[
                        "name"
                    ]
                )

                content = (
                    item[
                        "content"
                    ]
                )


            print(
                "Uploaded file:",
                filename
            )


            # --------------------------------------------------------------
            # Validate filename
            # --------------------------------------------------------------

            if (
                filename
                !=
                "AnimeSR_v2.pth"
            ):

                print(
                    "\n⚠ File name is not AnimeSR_v2.pth"
                )

                print(
                    "Current name:",
                    filename
                )


            # --------------------------------------------------------------
            # Save temporary upload
            # --------------------------------------------------------------

            temp_path = (
                UPLOAD_DIR
                /
                filename
            )


            with open(
                temp_path,
                "wb"
            ) as f:

                f.write(
                    bytes(
                        content
                    )
                )


            print(
                "Saved upload:",
                temp_path
            )


            print(
                "Uploaded size:",
                f"{temp_path.stat().st_size / 1024**2:.2f} MB"
            )


            # --------------------------------------------------------------
            # Basic validation
            # --------------------------------------------------------------

            if (
                temp_path.stat().st_size
                <
                1024 * 1024
            ):

                raise RuntimeError(
                    "File .pth quá nhỏ, có thể upload lỗi."
                )


            # --------------------------------------------------------------
            # Copy to AnimeSR weights
            # --------------------------------------------------------------

            shutil.copy2(
                temp_path,
                ANIMESR_TARGET
            )


            ANIMESR_CKPT = (
                ANIMESR_TARGET
            )


            print(
                "\n✓ AnimeSR_v2.pth ready"
            )

            print(
                "Final path:",
                ANIMESR_CKPT
            )

            print(
                "Final size:",
                f"{ANIMESR_CKPT.stat().st_size / 1024**2:.2f} MB"
            )


    # ==========================================================================
    # 7. REGISTER CALLBACK
    # ==========================================================================

    uploader_animesr.observe(
        save_animesr_weight,
        names="value"
    )


    # ==========================================================================
    # 8. SHOW WIDGET
    # ==========================================================================

    display(
        uploader_animesr
    )

    display(
        upload_output
    )

## Cell 8 — Tìm các sequence trong testLQ

Tạo testLQ

In [ ]:
# ==============================================================================
# CREATE testLQ FROM test_GT
# ==============================================================================

from pathlib import Path
import math
import random

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


# ------------------------------------------------------------------------------
# PATH
# ------------------------------------------------------------------------------

DATA_ROOT = Path(
    "/kaggle/input/datasets/duonghuyle/realbasicvsr-anime-data-v1/dataset_final_3"
)

TEST_GT = (
    DATA_ROOT /
    "test_GT"
)

TEST_LQ = Path(
    "/kaggle/working/testLQ"
)

TEST_LQ.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------------------

SCALE = 4

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# Degradation cố định để benchmark có thể tái lập.
BLUR_SIGMA = 0.8
NOISE_STD = 0.015

SEED = 1234


print("TEST_GT:", TEST_GT)
print("TEST_LQ:", TEST_LQ)
print("DEVICE :", DEVICE)


assert TEST_GT.exists(), (
    f"Không tìm thấy {TEST_GT}"
)


# ==============================================================================
# GAUSSIAN KERNEL
# ==============================================================================

def gaussian_kernel_2d(
    k=5,
    sigma=0.8,
    device="cuda"
):

    d = (
        k - 1
    ) / 2.0


    coords = (
        torch.arange(
            k,
            device=device,
            dtype=torch.float32
        )
        - d
    )


    y, x = torch.meshgrid(
        coords,
        coords,
        indexing="ij"
    )


    kernel = torch.exp(
        -(
            x ** 2
            +
            y ** 2
        )
        /
        (
            2.0
            *
            sigma ** 2
        )
    )


    # Quan trọng:
    # normalize kernel 2D trước khi repeat RGB.
    kernel = (
        kernel
        /
        kernel.sum()
    )


    return kernel


# ==============================================================================
# YUV420 CHROMA SUBSAMPLING
# ==============================================================================

def apply_chroma_subsample(
    x
):

    # x:
    # N,C,H,W
    # normalized [0,1]

    r = x[:, 0:1]
    g = x[:, 1:2]
    b = x[:, 2:3]


    y = (
        0.299
        * r

        +
        0.587
        * g

        +
        0.114
        * b
    )


    cb = (
        -0.1687
        * r

        -
        0.3313
        * g

        +
        0.5
        * b

        +
        0.5
    )


    cr = (
        0.5
        * r

        -
        0.4187
        * g

        -
        0.0813
        * b

        +
        0.5
    )


    h, w = (
        cb.shape[2:]
    )


    # 4:2:0
    cb_small = F.interpolate(
        cb,
        size=(
            max(1, h // 2),
            max(1, w // 2)
        ),
        mode="bilinear",
        align_corners=False
    )


    cr_small = F.interpolate(
        cr,
        size=(
            max(1, h // 2),
            max(1, w // 2)
        ),
        mode="bilinear",
        align_corners=False
    )


    cb = F.interpolate(
        cb_small,
        size=(h, w),
        mode="bilinear",
        align_corners=False
    )


    cr = F.interpolate(
        cr_small,
        size=(h, w),
        mode="bilinear",
        align_corners=False
    )


    r_new = (
        y
        +
        1.402
        *
        (
            cr
            -
            0.5
        )
    )


    g_new = (
        y

        -
        0.3441
        *
        (
            cb
            -
            0.5
        )

        -
        0.7141
        *
        (
            cr
            -
            0.5
        )
    )


    b_new = (
        y

        +
        1.772
        *
        (
            cb
            -
            0.5
        )
    )


    return torch.clamp(
        torch.cat(
            [
                r_new,
                g_new,
                b_new
            ],
            dim=1
        ),
        0.0,
        1.0
    )


# ==============================================================================
# DEGRADE ONE FRAME
# ==============================================================================

@torch.no_grad()
def make_lq_frame(
    rgb,
    seed
):

    # ----------------------------------------------------------
    # RGB uint8 -> Tensor
    # ----------------------------------------------------------

    x = (
        torch.from_numpy(
            rgb
        )
        .permute(
            2,
            0,
            1
        )
        .float()
        .unsqueeze(0)
        /
        255.0
    ).to(
        DEVICE
    )


    _, C, H, W = (
        x.shape
    )


    # ----------------------------------------------------------
    # Crop biên để chia hết x4
    # ----------------------------------------------------------

    H4 = (
        H // SCALE
    ) * SCALE

    W4 = (
        W // SCALE
    ) * SCALE


    x = x[
        :,
        :,
        :H4,
        :W4
    ]


    # ----------------------------------------------------------
    # 1. Gaussian blur
    # ----------------------------------------------------------

    k2d = gaussian_kernel_2d(
        k=5,
        sigma=BLUR_SIGMA,
        device=DEVICE
    )


    kernel = (
        k2d
        .reshape(
            1,
            1,
            5,
            5
        )
        .repeat(
            C,
            1,
            1,
            1
        )
    )


    x = F.conv2d(
        x,
        kernel,
        padding=2,
        groups=C
    )


    # ----------------------------------------------------------
    # 2. Bicubic downsample x4
    # ----------------------------------------------------------

    x = F.interpolate(
        x,
        scale_factor=(
            1.0
            /
            SCALE
        ),
        mode="bicubic",
        align_corners=False
    )


    # ----------------------------------------------------------
    # 3. YUV420
    # ----------------------------------------------------------

    x = apply_chroma_subsample(
        x
    )


    # ----------------------------------------------------------
    # 4. Gaussian noise cố định
    # ----------------------------------------------------------

    generator = (
        torch.Generator(
            device=DEVICE
        )
    )


    generator.manual_seed(
        seed
    )


    noise = torch.randn(
        x.shape,
        generator=generator,
        device=DEVICE,
        dtype=x.dtype
    )


    x = (
        x
        +
        noise
        *
        NOISE_STD
    )


    x = torch.clamp(
        x,
        0.0,
        1.0
    )


    # ----------------------------------------------------------
    # Tensor -> RGB uint8
    # ----------------------------------------------------------

    out = (
        x[0]
        .permute(
            1,
            2,
            0
        )
        .cpu()
        .numpy()
    )


    out = (
        out
        *
        255.0
    ).round().astype(
        np.uint8
    )


    return out


# ==============================================================================
# CREATE testLQ
# ==============================================================================

clips = sorted([
    p
    for p in TEST_GT.iterdir()
    if p.is_dir()
])


print(
    "Clips:",
    len(clips)
)


total_frames = 0


for clip_idx, clip_dir in enumerate(
    tqdm(
        clips,
        desc="Creating testLQ clips"
    )
):

    out_dir = (
        TEST_LQ
        /
        clip_dir.name
    )


    out_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    frame_files = sorted(
        clip_dir.glob(
            "*.png"
        )
    )


    for frame_idx, frame_path in enumerate(
        frame_files
    ):

        bgr = cv2.imread(
            str(
                frame_path
            ),
            cv2.IMREAD_COLOR
        )


        if bgr is None:

            raise RuntimeError(
                f"Không đọc được {frame_path}"
            )


        rgb = cv2.cvtColor(
            bgr,
            cv2.COLOR_BGR2RGB
        )


        # Seed khác nhau nhưng cố định cho từng frame
        seed = (
            SEED
            +
            clip_idx * 1000
            +
            frame_idx
        )


        lq = make_lq_frame(
            rgb,
            seed
        )


        save_path = (
            out_dir
            /
            frame_path.name
        )


        cv2.imwrite(
            str(
                save_path
            ),
            cv2.cvtColor(
                lq,
                cv2.COLOR_RGB2BGR
            )
        )


        total_frames += 1


print(
    "\n✓ testLQ created"
)

print(
    "Path:",
    TEST_LQ
)

print(
    "Frames:",
    total_frames
)

In [ ]:
def image_files(folder):

    return sorted([
        p

        for p in Path(
            folder
        ).iterdir()

        if (
            p.is_file()
            and
            p.suffix.lower()
            in IMG_EXT
        )
    ])


def discover_sequences(root):

    root = Path(root)


    direct = image_files(
        root
    )


    if direct:

        return [
            (
                root.name,
                root
            )
        ]


    sequences = []


    for d in sorted(
        p
        for p in root.iterdir()
        if p.is_dir()
    ):

        if image_files(d):

            sequences.append(
                (
                    d.name,
                    d
                )
            )


    return sequences


sequences = (
    discover_sequences(
        TEST_LQ
    )
)


print(
    "Sequences:",
    len(sequences)
)


for name, folder in (
    sequences[:10]
):

    print(
        name,
        len(
            image_files(
                folder
            )
        )
    )

## Cell 9 — Test 2 RealBasicVSR trên testLQ

In [ ]:
TEST_OUT = (
    OUT_ROOT /
    "testLQ"
)

RB_ORIG_OUT = (
    TEST_OUT /
    "realbasicvsr_original"
)

RB_FT_OUT = (
    TEST_OUT /
    "realbasicvsr_finetuned"
)


runtime_rows = []


for (
    seq_name,
    folder
) in sequences:

    files = image_files(
        folder
    )


    frames = [
        read_rgb(p)
        for p in files
    ]


    print(
        "\nSequence:",
        seq_name
    )


    # -------------------------
    # Original
    # -------------------------

    t0 = time.time()


    sr_original = (
        infer_rb_sequence(
            rb_original,
            frames,
            CHUNK,
            OVERLAP
        )
    )


    time_original = (
        time.time()
        - t0
    )


    # -------------------------
    # Fine-tuned
    # -------------------------

    t0 = time.time()


    sr_finetuned = (
        infer_rb_sequence(
            rb_finetuned,
            frames,
            CHUNK,
            OVERLAP
        )
    )


    time_ft = (
        time.time()
        - t0
    )


    # -------------------------
    # Save
    # -------------------------

    for p, img in zip(
        files,
        sr_original
    ):

        save_rgb(
            RB_ORIG_OUT
            / seq_name
            / p.name,
            img
        )


    for p, img in zip(
        files,
        sr_finetuned
    ):

        save_rgb(
            RB_FT_OUT
            / seq_name
            / p.name,
            img
        )


    runtime_rows.append({

        "sequence":
            seq_name,

        "frames":
            len(frames),

        "rb_original_sec":
            time_original,

        "rb_finetuned_sec":
            time_ft
    })


runtime_df = pd.DataFrame(
    runtime_rows
)

display(
    runtime_df
)

## Cell 10 — AnimeSR trên testLQ

In [ ]:
!pip install ffmpeg

In [ ]:
from pathlib import Path

# Kaggle path hiện tại của BasicSR
degradation_file = Path(
    "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"
)

assert degradation_file.exists(), degradation_file

text = degradation_file.read_text()

old = (
    "from torchvision.transforms.functional_tensor "
    "import rgb_to_grayscale"
)

new = (
    "from torchvision.transforms.functional "
    "import rgb_to_grayscale"
)

if old in text:
    text = text.replace(old, new)
    degradation_file.write_text(text)

    print("✓ Patched BasicSR torchvision import")
else:
    print("Import cũ không còn tồn tại hoặc đã được patch.")

print("\nCurrent line:")
for line in degradation_file.read_text().splitlines():
    if "rgb_to_grayscale" in line:
        print(line)

In [ ]:
ANIME_TEST_OUT = (
    TEST_OUT /
    "animesr"
)

cmd = [

    sys.executable,

    str(
        ANIMESR_REPO
        /
        "scripts/"
        "inference_animesr_frames.py"
    ),

    "-i",
    str(TEST_LQ),

    "-n",
    "AnimeSR_v2",

    "-s",
    "4",

    "-o",
    str(
        ANIME_TEST_OUT
    ),

    "--expname",
    "animesr_v2"
]

print(
    " ".join(cmd)
)

subprocess.run(
    cmd,
    cwd=str(ANIMESR_REPO),
    check=True
)

## Cell 11 — Visual comparison testLQ
LQ Bicubic x4
RealBasicVSR original
RealBasicVSR fine-tuned
AnimeSR v2

LQ bicubic không phải model competitor, chỉ là reference để biết input ban đầu xấu thế nào.

In [ ]:
def find_anime_output(
    filename,
    seq_name
):

    all_imgs = [

        p
        for p in (
            ANIME_TEST_OUT
            .rglob("*")
        )

        if (
            p.is_file()
            and
            p.suffix.lower()
            in IMG_EXT
        )
    ]


    # ưu tiên đúng clip
    hits = [

        p
        for p in all_imgs

        if (
            p.name
            ==
            filename

            and

            seq_name
            in str(p)
        )
    ]


    if not hits:

        hits = [

            p
            for p in all_imgs

            if p.name
            ==
            filename
        ]


    return (
        hits[0]
        if hits
        else None
    )

Plot:

In [ ]:
def show_comparison(
    seq_name,
    folder,
    n_show=4
):

    files = image_files(
        folder
    )


    indices = np.linspace(
        0,
        len(files) - 1,
        min(
            n_show,
            len(files)
        )
    ).astype(int)


    fig, axes = plt.subplots(
        len(indices),
        4,
        figsize=(
            20,
            5 * len(indices)
        )
    )


    if len(indices) == 1:

        axes = np.expand_dims(
            axes,
            axis=0
        )


    for row, idx in enumerate(
        indices
    ):

        p = files[idx]


        lq = read_rgb(
            p
        )


        h, w = (
            lq.shape[:2]
        )


        bicubic = cv2.resize(
            lq,
            (
                w * 4,
                h * 4
            ),
            interpolation=(
                cv2.INTER_CUBIC
            )
        )


        rb_o = read_rgb(
            RB_ORIG_OUT
            / seq_name
            / p.name
        )


        rb_f = read_rgb(
            RB_FT_OUT
            / seq_name
            / p.name
        )


        anime_path = (
            find_anime_output(
                p.name,
                seq_name
            )
        )


        anime = (
            read_rgb(
                anime_path
            )
            if anime_path
            else np.zeros_like(
                rb_f
            )
        )


        images = [
            (
                bicubic,
                "LQ Bicubic ×4"
            ),
            (
                rb_o,
                "RealBasicVSR original"
            ),
            (
                rb_f,
                "RealBasicVSR fine-tuned"
            ),
            (
                anime,
                "AnimeSR v2"
            )
        ]


        for col, (
            image,
            title
        ) in enumerate(images):

            axes[
                row,
                col
            ].imshow(image)

            axes[
                row,
                col
            ].set_title(
                title
            )

            axes[
                row,
                col
            ].axis("off")


    plt.tight_layout()


    out = (
        TEST_OUT
        /
        f"comparison_{seq_name}.png"
    )


    plt.savefig(
        out,
        dpi=160,
        bbox_inches="tight"
    )


    plt.show()


    print(
        "Saved:",
        out
    )

Chạy:

In [ ]:
for (
    name,
    folder
) in sequences[:3]:

    show_comparison(
        name,
        folder,
        n_show=4
    )

## Cell 12 — Video upload hoặc tải Internet

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

UPLOAD_DIR = Path("/kaggle/working/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

uploader = widgets.FileUpload(
    accept=".mp4,.mkv,.mov,.avi,.webm",
    multiple=False
)

output = widgets.Output()

def save_uploaded_file(change):
    global VIDEO_PATH

    if len(uploader.value) == 0:
        return

    with output:
        output.clear_output()

        if isinstance(uploader.value, dict):
            item = next(iter(uploader.value.values()))
            filename = item["metadata"]["name"]
            content = item["content"]
        else:
            item = uploader.value[0]
            filename = item["name"]
            content = item["content"]

        VIDEO_PATH = UPLOAD_DIR / filename

        with open(VIDEO_PATH, "wb") as f:
            f.write(bytes(content))

        print("✓ Upload thành công")
        print("VIDEO_PATH:", VIDEO_PATH)
        print(
            "Size:",
            f"{VIDEO_PATH.stat().st_size / 1024**2:.2f} MB"
        )

uploader.observe(
    save_uploaded_file,
    names="value"
)

display(uploader)
display(output)

## Cell 13 — Đọc video

Không resize video xuống max_side=160.

In [ ]:
def read_video(
    path,
    max_frames=None
):

    cap = cv2.VideoCapture(
        str(path)
    )


    fps = cap.get(
        cv2.CAP_PROP_FPS
    )


    if (
        fps is None
        or
        fps <= 0
    ):

        fps = 24.0


    frames = []


    while True:

        ok, bgr = cap.read()


        if not ok:

            break


        rgb = cv2.cvtColor(
            bgr,
            cv2.COLOR_BGR2RGB
        )


        frames.append(
            rgb
        )


        if (
            max_frames
            is not None

            and

            len(frames)
            >= max_frames
        ):

            break


    cap.release()


    return (
        frames,
        float(fps)
    )

Chạy:

In [ ]:
video_frames, fps = (
    read_video(
        VIDEO_PATH,
        MAX_VIDEO_FRAMES
    )
)


print(
    "Frames:",
    len(video_frames)
)

print(
    "FPS:",
    fps
)

print(
    "LQ size:",
    video_frames[0].shape
)

## Cell 14 — RealBasicVSR original vs fine-tuned trên video

In [ ]:
VIDEO_OUT = (
    OUT_ROOT /
    "video"
)

VIDEO_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Original
# ------------------------------------------------------------

t0 = time.time()


video_original = (
    infer_rb_sequence(
        rb_original,
        video_frames,
        CHUNK,
        OVERLAP
    )
)


time_original = (
    time.time()
    - t0
)


# ------------------------------------------------------------
# Fine-tuned
# ------------------------------------------------------------

t0 = time.time()


video_finetuned = (
    infer_rb_sequence(
        rb_finetuned,
        video_frames,
        CHUNK,
        OVERLAP
    )
)


time_finetuned = (
    time.time()
    - t0
)


print(
    "RB original:",
    f"{time_original:.1f}s"
)

print(
    "RB fine-tuned:",
    f"{time_finetuned:.1f}s"
)

## Cell 15 — AnimeSR trên video

In [ ]:
# ==============================================================================
# FIX ffmpeg-python CONFLICT FOR ANIMESR
# ==============================================================================

import sys
import subprocess

print("Removing conflicting ffmpeg packages...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "ffmpeg",
        "python-ffmpeg"
    ],
    check=False
)

print("\nInstalling correct ffmpeg-python...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--force-reinstall",
        "ffmpeg-python==0.2.0"
    ],
    check=True
)

print("\n✓ ffmpeg-python installed")

In [ ]:
# ==============================================================================
# CELL 15 - ANIMESR VIDEO INFERENCE (SAFE SINGLE-GPU MODE)
# ==============================================================================

from pathlib import Path
import os
import sys
import subprocess
import shutil


ANIME_VIDEO_OUT = (
    VIDEO_OUT /
    "animesr"
)

ANIME_VIDEO_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------------------------
# Clean previous failed output
# ------------------------------------------------------------------------------

EXP_DIR = (
    ANIME_VIDEO_OUT /
    "animesr_v2"
)

if EXP_DIR.exists():

    print(
        "Removing previous failed AnimeSR output..."
    )

    shutil.rmtree(
        EXP_DIR
    )


# ------------------------------------------------------------------------------
# Force AnimeSR subprocess to see ONLY GPU 0
# ------------------------------------------------------------------------------

env = os.environ.copy()

env[
    "CUDA_VISIBLE_DEVICES"
] = "0"


# ------------------------------------------------------------------------------
# Build command
# ------------------------------------------------------------------------------

cmd = [

    sys.executable,

    str(
        ANIMESR_REPO
        /
        "scripts"
        /
        "inference_animesr_video.py"
    ),

    "-i",
    str(
        VIDEO_PATH
    ),

    "-n",
    "AnimeSR_v2",

    "-s",
    "4",

    "-o",
    str(
        ANIME_VIDEO_OUT
    ),

    "--expname",
    "animesr_v2",

    "--num_process_per_gpu",
    "1",

    "--suffix",
    "benchmark"
]


print(
    "Command:"
)

print(
    " ".join(cmd)
)

print(
    "\nAnimeSR visible GPUs: 1"
)


# ------------------------------------------------------------------------------
# Run
# ------------------------------------------------------------------------------

result = subprocess.run(
    cmd,
    cwd=str(
        ANIMESR_REPO
    ),
    env=env,
    text=True
)


print(
    "\nReturn code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "AnimeSR video inference failed."
    )


# ------------------------------------------------------------------------------
# Find generated video
# ------------------------------------------------------------------------------

anime_videos = sorted(
    ANIME_VIDEO_OUT.rglob(
        "*.mp4"
    )
)


print(
    "\nGenerated videos:"
)

for p in anime_videos:

    print(
        p,
        f"{p.stat().st_size / 1024**2:.2f} MB"
    )


assert anime_videos, (
    "AnimeSR command kết thúc nhưng "
    "không tạo được video output."
)


ANIME_VIDEO_PATH = (
    anime_videos[-1]
)


print(
    "\n✓ AnimeSR video ready:"
)

print(
    ANIME_VIDEO_PATH
)

## Cell 16 — Hàm ghi video

In [ ]:
def write_video(
    path,
    frames,
    fps
):

    path = Path(path)


    h, w = (
        frames[0]
        .shape[:2]
    )


    writer = cv2.VideoWriter(
        str(path),
        cv2.VideoWriter_fourcc(
            *"mp4v"
        ),
        fps,
        (
            w,
            h
        )
    )


    for rgb in frames:

        writer.write(
            cv2.cvtColor(
                rgb,
                cv2.COLOR_RGB2BGR
            )
        )


    writer.release()

Save

In [ ]:
write_video(
    VIDEO_OUT /
    "realbasicvsr_original.mp4",
    video_original,
    fps
)


write_video(
    VIDEO_OUT /
    "realbasicvsr_finetuned.mp4",
    video_finetuned,
    fps
)

## Cell 17 — Runtime comparison

In [ ]:
runtime = pd.DataFrame([

    {
        "model":
            "RealBasicVSR Original",

        "frames":
            len(video_frames),

        "seconds":
            time_original,

        "infer_fps":
            (
                len(video_frames)
                /
                time_original
            )
    },

    {
        "model":
            "RealBasicVSR Fine-tuned",

        "frames":
            len(video_frames),

        "seconds":
            time_finetuned,

        "infer_fps":
            (
                len(video_frames)
                /
                time_finetuned
            )
    }

])


display(
    runtime
)


runtime.to_csv(
    VIDEO_OUT /
    "runtime.csv",
    index=False
)

## tải kết quả test

In [ ]:
# ==============================================================================
# FINAL CELL - ZIP ALL TEST RESULTS
# ==============================================================================

from pathlib import Path
import shutil

RESULT_DIR = Path(
    "/kaggle/working/vsr_benchmark"
)

ZIP_BASE = Path(
    "/kaggle/working/RealBasicVSR_AnimeSR_Test_Results"
)

assert RESULT_DIR.exists(), (
    f"Không tìm thấy thư mục kết quả: {RESULT_DIR}"
)

print("Zipping test results...")

zip_path = shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=str(RESULT_DIR)
)

zip_path = Path(zip_path)

print("\n✓ ZIP created")
print("Path :", zip_path)
print(
    "Size :",
    f"{zip_path.stat().st_size / 1024**2:.2f} MB"
)

print("\nIncluded files:")

for p in sorted(
    RESULT_DIR.rglob("*")
):
    if p.is_file():
        print(
            p.relative_to(
                RESULT_DIR
            )
        )

In [ ]:
# ==============================================================================
# OPTIONAL - ZIP REPORT-ONLY RESULTS
# ==============================================================================

from pathlib import Path
import zipfile

ROOT = Path(
    "/kaggle/working/vsr_benchmark"
)

ZIP_PATH = Path(
    "/kaggle/working/RealBasicVSR_AnimeSR_Test_Report.zip"
)

KEEP_EXT = {
    ".csv",
    ".png",
    ".mp4",
    ".txt"
}

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for p in ROOT.rglob("*"):

        if (
            p.is_file()
            and
            p.suffix.lower()
            in KEEP_EXT
        ):

            zf.write(
                p,
                arcname=p.relative_to(ROOT)
            )

print("✓ Created:", ZIP_PATH)
print(
    "Size:",
    f"{ZIP_PATH.stat().st_size / 1024**2:.2f} MB"
)

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display

ZIP_PATH = Path(
    "/kaggle/working/RealBasicVSR_AnimeSR_Test_Results.zip"
)

assert ZIP_PATH.exists(), (
    f"Không tìm thấy: {ZIP_PATH}"
)

print(
    "ZIP size:",
    f"{ZIP_PATH.stat().st_size / 1024**2:.2f} MB"
)

display(
    FileLink(
        str(ZIP_PATH),
        result_html_prefix="👉 Bấm vào đây để tải ZIP: "
    )
)